# Recalculate `data.csv` features

This notebook rebuilds every calculated feature column for every row in `data.csv`. It preserves the row metadata and recalculates the 28 features for all cells, pyramidal cells, inhibitory interneurons, and excitatory interneurons from each row's `source_file`.

The feature calculations live in `src.utils.calculate_simulation_features`; the notebook is only responsible for reading the table, passing each row's analysis window to that function, comparing the result, and writing a new CSV. It never overwrites `data.csv`.

Before running it, extract `data.zip` into the project root so paths such as `data/sim7Hz0.25Vm/sim_with_stim.json` exist, and use the environment created from `requirements.txt`. If needed, run this once before the notebook:

```python
from zipfile import ZipFile
with ZipFile("data.zip") as archive:
    archive.extractall(".")
```

In [15]:
from pathlib import Path

import numpy as np
import pandas as pd

from src.utils import FEATURE_COLUMNS, calculate_simulation_features

PROJECT_ROOT = Path.cwd()
INPUT_CSV = PROJECT_ROOT / "data.csv"
OUTPUT_CSV = PROJECT_ROOT / "data_recalculated.csv"

if not INPUT_CSV.is_file():
    raise FileNotFoundError(f"Could not find {INPUT_CSV}")

source_data = pd.read_csv(INPUT_CSV)
missing_feature_columns = set(FEATURE_COLUMNS).difference(source_data.columns)
if missing_feature_columns:
    raise ValueError(
        "data.csv does not contain the expected feature columns: "
        + ", ".join(sorted(missing_feature_columns))
    )

print(f"Loaded {len(source_data)} rows from {INPUT_CSV.name}")
print(f"Will write the recalculated table to {OUTPUT_CSV.name}")

Loaded 55 rows from data.csv
Will write the recalculated table to data_recalculated.csv


In [16]:
source_paths = source_data["source_file"].map(
    lambda value: Path(value) if Path(value).is_absolute() else PROJECT_ROOT / Path(value)
)
exists = source_paths.map(Path.is_file)
if not exists.any():
    raise FileNotFoundError("No source_file paths exist. Extract data.zip into the project root first.")
if not exists.all():
    preview = "\n".join(f"- {path}" for path in source_paths[~exists])
    print(f"Skipping {(~exists).sum()} row(s) with missing source_file paths (data.csv is not modified):\n{preview}")
    source_data = source_data[exists]
    source_paths = source_paths[exists]

## Recalculate the feature columns

Each row retains its original metadata and analysis window. `delay` is passed as the stimulation onset reference, matching the original import workflow. The utility function recalculates `plv`, `ppc`, `deltafr`, `complex`, `amplitude_norm`, `power_norm`, and `total_power` for all four cell groups.

In [17]:
recalculated_records = []
failures = []

for row_number, (row_index, row) in enumerate(source_data.iterrows(), start=1):
    config_path = source_paths.loc[row_index]
    sim_id = row["sim_id"]
    print(f"[{row_number}/{len(source_data)}] {sim_id}: {config_path}")

    try:
        duration = float(row['analysis_stop']) - float(row['analysis_start'])
        half_duration = duration / 2
        midpoint = float(row['analysis_start']) + half_duration
        feature_values = calculate_simulation_features(
            config_path=config_path,
            analysis_start=float(row['analysis_start']),
            analysis_stop=float(row['analysis_stop']),
            true_delay=float(row['delay']),
        )
    except Exception as error:
        failures.append({
            "row": row_index,
            "sim_id": sim_id,
            "source_file": str(config_path),
            "error": str(error),
        })
        continue

    record = row.to_dict()
    record.update(feature_values)
    recalculated_records.append(record)

if failures:
    failure_report = pd.DataFrame(failures)
    display(failure_report)
    raise RuntimeError(
        f"Feature extraction failed for {len(failures)} simulation(s); no output CSV was written."
    )

recalculated_data = pd.DataFrame(recalculated_records, columns=source_data.columns)
assert len(recalculated_data) == len(source_data)
recalculated_data.head()

[1/55] baseline: /home/habek/ti-coupling-project/data/sim0_CoreNeuron/sim_with_stim.json
0.0
0.0
0.0
0.0
[2/55] highCorrelation_baseline: /home/habek/ti-coupling-project/data/highCorrelation_convergence/sim_with_stim.json
0.0
0.0
0.0
0.0
[3/55] baseline_highCorrelation_stopStart: /home/habek/ti-coupling-project/data/highCorrelation_convergence_stopStart/sim_with_stim.json
0.0
0.0
0.0
0.0
[4/55] baseline_highCorrelation_stopStart_short: /home/habek/ti-coupling-project/data/highCorrelation_convergence_stopStart_short/sim_with_stim.json
0.0
0.0
0.0
0.0
[5/55] resting_3000: /home/habek/ti-coupling-project/data/resting_3000/sim_with_stim.json
0.0
0.0
0.0
0.0
[6/55] resting: /home/habek/ti-coupling-project/data/resting/sim_with_stim.json
0.0
0.0
0.0
0.0
[7/55] 7Hz0.25Vm: /home/habek/ti-coupling-project/data/sim7Hz0.25Vm/sim_with_stim.json
-0.2383270843787174
-0.38431899452794543
2.249831363162091
1.790899405049741
[8/55] 7Hz0.25Vm_seed2: /home/habek/ti-coupling-project/data/sim7Hz0.25Vm_seed

,sim_id,time_import,dt,tstop,seed,simulator,node_set,delay,duration,stim_freq,...,total_power_INT_INH,plv_INT_EXC,ppc_INT_EXC,deltafr_INT_EXC,complex_INT_EXC,amplitude_norm_INT_EXC,power_norm_INT_EXC,total_power_INT_EXC,deployment,neurodamus
0,baseline,2026-05-18,0.025,2500,843,CORENEURON,hex0,0,0,0,...,0.191756,1.0+0.0j,1.0,NaN,11.111111+ 0.000000j,1.0,0.591080,0.005170,Mimas,bluebrain/neurodamus
1,highCorrelation_baseline,2026-05-21,0.025,2500,843,CORENEURON,hex0,0,0,0,...,1.066437,1.0+0.0j,1.0,NaN,37.681159+ 0.000000j,1.0,0.846024,0.041541,Mimas,bluebrain/neurodamus
2,baseline_highCorrelation_stopStart,2026-05-22,0.025,5000,843,CORENEURON,hex0,0,0,0,...,1.022241,1.0+0.0j,1.0,NaN,140.579710- 0.000000j,1.0,0.837130,0.045038,Mimas,bluebrain/neurodamus
3,baseline_highCorrelation_stopStart_short,2026-05-23,0.025,3000,843,CORENEURON,hex0,0,0,0,...,1.019818,1.0+0.0j,1.0,NaN,56.521739+ 0.000000j,1.0,0.808130,0.030503,Mimas,bluebrain/neurodamus
4,resting_3000,2026-09-15,0.025,3000,843,CORENEURON,hex0,0,0,0,...,0.201316,1.0+0.0j,1.0,NaN,49.758454+ 0.000000j,1.0,0.716163,0.013774,mimas,4.2


## Compare with the existing table

The comparison is tolerance-based because CSV serialisation and floating-point arithmetic can differ in their final digits. Complex values are compared as complex numbers.

In [21]:
comparison_rows = []
for column in FEATURE_COLUMNS:
    existing = source_data[column]
    recalculated = recalculated_data[column]

    if column.startswith("complex"):
        existing_values = existing.map(lambda value: np.nan if pd.isna(value) else complex(value))
        recalculated_values = recalculated.map(
            lambda value: np.nan if pd.isna(value) else complex(value)
        )
        matches = np.isclose(
            existing_values.to_numpy(),
            recalculated_values.to_numpy(),
            rtol=1e-4,
            atol=1e-5,
            equal_nan=True,
        )
    else:
        matches = np.isclose(
            pd.to_numeric(existing, errors="coerce").to_numpy(),
            pd.to_numeric(recalculated, errors="coerce").to_numpy(),
            rtol=1e-4,
            atol=1e-5,
            equal_nan=True,
        )

    comparison_rows.append({
        "feature": column,
        "matching_rows": int(matches.sum()),
        "different_rows": int((~matches).sum()),
        "row_index": (~matches).nonzero()[0].tolist()
    })

comparison = pd.DataFrame(comparison_rows)
display(comparison)
print(f"Total differing feature values: {comparison['different_rows'].sum()}")

,feature,matching_rows,different_rows,row_index
0,plv,0,55,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
1,ppc,55,0,[]
2,deltafr,55,0,[]
3,complex,55,0,[]
4,amplitude_norm,55,0,[]
5,power_norm,55,0,[]
6,total_power,55,0,[]
7,plv_PYR,0,55,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
8,ppc_PYR,55,0,[]
9,deltafr_PYR,55,0,[]


Total differing feature values: 214


In [12]:
#sort base on freq and inside freq based on stim_amp
recalculated_data.sort_values(by=["stim_freq", "stim_amp"], inplace=True)

In [20]:
source_data['plv']

0                           (0.9999999999999999+0j)
1                           (0.9999999999999999+0j)
2                           (0.9999999999999999+0j)
3                                            (1+0j)
4                                            (1+0j)
5                                            (1+0j)
6      (0.021665062073351775-0.005263403911089929j)
7         (0.006474758290008912+0.014234089881253j)
8       (0.02686539894549462-0.002348885401817571j)
9     (0.037599315239719676-0.0072807243095361065j)
10     (0.011869203246630346+0.020331167255259124j)
11     (0.011869203246630346+0.020331167255259124j)
12      (0.02308396741843509+0.033083108147119115j)
13      (0.04694942936382634+0.005349841633103135j)
14      (0.04694942936382634+0.005349841633103135j)
15       (0.03330886062633066+0.05713686613171697j)
16      (0.05313603162669844+0.016169892978390752j)
17       (0.08598103262648728+0.04517653498724999j)
18      (0.017109907292937997+0.11568409577150093j)
19        (0

In [13]:
recalculated_data['plv']

0     1.000000+0.000000j
1     1.000000+0.000000j
2     1.000000+0.000000j
3     1.000000+0.000000j
4     1.000000+0.000000j
5     1.000000+0.000000j
6     0.021665-0.005263j
7     0.006475+0.014234j
8     0.026865-0.002349j
9     0.037599-0.007281j
10    0.011869+0.020331j
11    0.011869+0.020331j
12    0.023084+0.033083j
13    0.046949+0.005350j
14    0.046949+0.005350j
15    0.033309+0.057137j
16    0.053136+0.016170j
17    0.085981+0.045177j
18    0.017110+0.115684j
19    0.149947+0.134671j
20    0.170637+0.139910j
21    0.100600+0.266415j
22    0.170637+0.139910j
23    0.019812+0.008160j
24    0.010024+0.003258j
25    0.019264+0.003753j
26    0.009149+0.011971j
27    0.010385+0.047455j
28    0.014951+0.010032j
29    0.033314+0.024887j
30    0.025218+0.057436j
31    0.039244+0.019798j
32    0.038224+0.030553j
33    0.039158+0.036108j
34    0.034331+0.034225j
35    0.040738+0.037443j
36    0.049881+0.019333j
37    0.044553+0.048412j
38    0.065563+0.049575j
39    0.044713+0.094903j


## Write the regenerated table

This writes a separate file. Review `comparison` before choosing to replace the original `data.csv` yourself.

In [14]:
recalculated_data.to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {len(recalculated_data)} recalculated rows to {OUTPUT_CSV}")

Wrote 55 recalculated rows to /home/habek/ti-coupling-project/data_recalculated.csv
